# Drift monitoring + conditional retrain

The teaching demos (`run_demo`, `my_territory_demo`) retrain from scratch each run for
self-containment. That is **not** production ops. This notebook walks the real pattern
introduced in Phase 18:

1. **Serve** with a frozen deployed model.
2. **Monitor** accumulated logs for drift signals (reward PSI, calibration Δ, feature PSI,
   overlap health, rolling DR drop).
3. **Retrain only when signals fire** — never on a blind cron.
4. **Gate** the candidate through the same `PromotionGate` as Phase 5/17.
5. **Audit** every promote/hold decision append-only.

It reuses `scripts/simulate_drift_demo.py` to drive the loop end-to-end on seeded simulated drift,
then plots the four signature charts: reward histogram, calibration MAE over shifts, drift
signal time series with threshold lines, and the regret curve (frozen → trigger → post-retrain).


In [ ]:
import sys
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt

# Allow `import run_demo` and `import simulate_drift_demo` from scripts/.
ROOT = Path.cwd().resolve()
while ROOT.name != "nba" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from simulate_drift_demo import run_drift_demo  # noqa: E402


## 1. Run the drift demo

Small parameters keep the notebook responsive. The same seed reproduces the JSONL artifacts
under `artifacts/monitoring/` (drift reports + retrain audit) and the
`artifacts/drift_demo_report.json` summary.


In [ ]:
report = run_drift_demo(
    n_pre=1500,
    n_post=500,
    shifts=4,
    seed=7,
    report_path=ROOT / "artifacts" / "drift_demo_report.json",
)
print(f"promoted={report.promoted}  at shift={report.promote_shift}")
print(f"{len(report.shift_records)} shift records (frozen + post_retrain)")

## 2. Reward distribution — reference vs recent

Under drift the realized reward mass shifts toward APPOINTMENT/CLOSED (the spec lifts those
outcomes). The frozen model was calibrated to the pre-drift mix, so its `q(x,a)` lags the new
world. The `reward_psi` signal quantifies this shift.


In [ ]:
frozen = [s for s in report.shift_records if s.phase == "frozen"]
post = [s for s in report.shift_records if s.phase == "post_retrain"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([s.mean_reward for s in frozen], "o-", label="frozen serve")
ax.plot([s.mean_reward for s in post], "s-", label="post-retrain serve")
ax.set_xlabel("shift index")
ax.set_ylabel("mean realized reward")
ax.set_title("Realized reward per shift — frozen degrades, retrain recovers")
ax.legend()
plt.tight_layout()


## 3. Calibration MAE — frozen vs after retrain

The calibration_drift signal tracks `mean(|q(x,a_logged) − r|)` on the recent batch. Under drift
the frozen model's calibration error climbs; once the monitor triggers a retrain, the candidate
(fit on reference∪recent) brings the error back down.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
all_calib = [s.calibration_mae for s in report.shift_records]
ax.plot(all_calib, "o-")
if report.promote_shift is not None:
    ax.axvline(report.promote_shift, color="red", linestyle="--", label="retrain promote")
ax.set_xlabel("shift index (frozen | post-retrain)")
ax.set_ylabel("calibration MAE")
ax.set_title("Calibration MAE — frozen model miscalibrates, retrain recovers")
ax.legend()
plt.tight_layout()


## 4. Drift signals over shifts + threshold lines

The five signals (`reward_psi`, `calibration_drift`, `feature_psi_max`, `overlap_health`,
`rolling_dr_drop`) are computed per shift. Each carries a configured threshold; a breach is the
trigger that mandates a retrain.


In [ ]:
signal_names = list(report.shift_records[0].signals.keys())
fig, ax = plt.subplots(figsize=(9, 5))
for name in signal_names:
    vals = [s.signals.get(name, 0.0) for s in report.shift_records]
    ax.plot(vals, "o-", label=name)
if report.promote_shift is not None:
    ax.axvline(report.promote_shift, color="red", linestyle="--", label="retrain promote")
ax.set_xlabel("shift index (frozen | post-retrain)")
ax.set_ylabel("signal value")
ax.set_title("Drift signals over shifts — trigger breach mandates retrain")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()


## 5. Regret curve — frozen → trigger → post-retrain

Decision regret is the value lost vs the oracle that knew the true prizes. Under drift the frozen
model's regret rises; after the retrain promote it recovers toward the pre-drift baseline.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
all_regret = [s.mean_regret for s in report.shift_records]
ax.plot(all_regret, "o-")
if report.promote_shift is not None:
    ax.axvline(report.promote_shift, color="red", linestyle="--", label="retrain promote")
ax.set_xlabel("shift index (frozen | post-retrain)")
ax.set_ylabel("mean regret per shift")
ax.set_title("Decision regret — frozen rises under drift, retrain recovers")
ax.legend()
plt.tight_layout()


## 6. Takeaways

- **Frozen serve + conditional retrain** is the production pattern the teaching demos only
  *simulate*.
- The monitor signals are computed **offline** on logged `(context, action, reward, propensity)`
  — no oracle, no protected/geo fields.
- The retrain candidate must clear the same `PromotionGate` (DR lower bound) as Phase 5/17 — a
  retrain that doesn't beat the deployed model is audited HOLD and the deployed manifest is
  left intact.
- The optional Grafana + Prometheus stack (Phase 18) visualizes the same signals `run_monitor.py`
  already writes to JSONL — it is a read-only view, never a second source of truth.
